In [12]:
from pathlib import Path
from tempfile import TemporaryDirectory
import tarfile
import pandas as pd
import awkward as ak
import ROOT
import pyarrow as pa
import numpy as np
import duckdb
import os
import sys
sys.path.append(str(Path(os.getcwd()).parent.absolute()))
import colorlog
logger = colorlog.getLogger()

from ringer_zero.root import read_ttree_as_pdf

In [2]:
TTREE_NAME = 'run_410000/HLT/EgammaMon/summary/events'

In [3]:
source_files = {
    'jf17': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/bkg/user.isilvafe.user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_2sigma_constraint_26.10.23_EXT0.tap_jf17_2sigma_5M_XYZ.root.tgz'
        ),
        'target': 1
    },
    'zee': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_EXT0.tap_zee_2sigma_5M_XYZ.root.tgz'
        ),
        'target': 0
    }
}
samples_per_batch = 100000
output_dir = Path('/media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/data.parquet')
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def is_safe_int8_cast(series: pd.Series):
    unique_values = series.unique().to_numpy()
    if np.any(np.logical_and(unique_values != 0, unique_values != 1)):
        raise ValueError(f"Series must contain only 1's and 0's for int8 casting, found {unique_values}")
    return True


INT8_CASTS = [
    'el_lhtight',
    'el_lhmedium',
    'el_lhloose',
    'el_lhvloose',
    'el_dnntight',
    'el_dnnmedium',
    'el_dnnloose',
    'mc_hasMC',
    'mc_isTruthElectronAny',
    'mc_isTruthElectronFromZ',
    'mc_isTruthElectronFromJpsiPrompt'
]

def dump_df(df, file_count, processed_rows, output_dir):
    logger.info(f'Processing Batch {file_count} (Rows {processed_rows} to {processed_rows + len(df)})')
    output_path = output_dir / f'data_{file_count:04d}.parquet'
    df['id'] = pd.Series(range(processed_rows, processed_rows + len(df)), dtype="uint64[pyarrow]")
    for col in INT8_CASTS:
        is_safe_int8_cast(df[col])
        df[col] = df[col].astype('bool[pyarrow]')
    df.to_parquet(output_path, index=False)
    return file_count + 1, processed_rows + len(df)

In [5]:
df_accumulator = pd.DataFrame()
file_count = 0
processed_rows = 0

for sample_name, sample_info in source_files.items():
    for root_tar in sample_info['datapath'].glob('*.root.tgz'):
        logger.info(f'Reading {root_tar} for sample {sample_name}')
        with TemporaryDirectory() as temp_dir:
            temp_dir_path = Path(temp_dir)
            with tarfile.open(root_tar, 'r:gz') as tar:
                tar.extractall(path=temp_dir_path)
            for root_file in temp_dir_path.glob('*.root'):
                logger.info(f'Reading {root_file} for sample {sample_name}')
                df = read_ttree_as_pdf(root_file, ttree_name='run_410000/HLT/EgammaMon/summary/events')
                df['target'] = sample_info['target']
                df['target'] = df['target'].astype('uint8[pyarrow]')
                df_accumulator = pd.concat([df_accumulator, df], ignore_index=True)
                if len(df_accumulator) >= samples_per_batch:
                    file_count, processed_rows = dump_df(
                        df_accumulator, file_count, processed_rows, output_dir
                    )
                    df_accumulator = pd.DataFrame()  # Reset accumulator

if len(df_accumulator):
    file_count, processed_rows = dump_df(
        df_accumulator, file_count, processed_rows, output_dir
    )

 [Sun, 15 Mar 2026 21:04:11] INFO Reading /media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/bkg/user.isilvafe.user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_2sigma_constraint_26.10.23_EXT0.tap_jf17_2sigma_5M_XYZ.root.tgz/user.isilvafe.35439386._000001.XYZ.root.tgz for sample jf17
/tmp/ipykernel_986425/4247997998.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=temp_dir_path)
 [Sun, 15 Mar 2026 21:04:11] INFO Reading /tmp/tmp3a5jceip/output.35439386._000003.root for sample jf17
 [Sun, 15 Mar 2026 21:04:11] INFO Reading /tmp/tmp3a5jceip/output.35439386._000004.root for sample jf17
 [Sun, 15 Mar 2026 21:04:11] INFO Reading /media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/bkg/user.isilvafe.user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_2sig

Processing Batch 0 (Rows 0 to 101869)


 [Sun, 15 Mar 2026 21:04:16] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000301.root for sample jf17
 [Sun, 15 Mar 2026 21:04:16] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000295.root for sample jf17
 [Sun, 15 Mar 2026 21:04:16] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000294.root for sample jf17
 [Sun, 15 Mar 2026 21:04:16] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000282.root for sample jf17
 [Sun, 15 Mar 2026 21:04:17] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000298.root for sample jf17
 [Sun, 15 Mar 2026 21:04:17] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000299.root for sample jf17
 [Sun, 15 Mar 2026 21:04:17] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000285.root for sample jf17
 [Sun, 15 Mar 2026 21:04:17] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000281.root for sample jf17
 [Sun, 15 Mar 2026 21:04:18] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000300.root for sample jf17
 [Sun, 15 Mar 2026 21:04:18] INFO Reading /tmp/tmp5wb3xo51/outpu

Processing Batch 1 (Rows 101869 to 203423)


 [Sun, 15 Mar 2026 21:04:19] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000280.root for sample jf17
 [Sun, 15 Mar 2026 21:04:19] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000289.root for sample jf17
 [Sun, 15 Mar 2026 21:04:20] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000297.root for sample jf17
 [Sun, 15 Mar 2026 21:04:20] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000279.root for sample jf17
 [Sun, 15 Mar 2026 21:04:20] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000278.root for sample jf17
 [Sun, 15 Mar 2026 21:04:20] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000286.root for sample jf17
 [Sun, 15 Mar 2026 21:04:21] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000292.root for sample jf17
 [Sun, 15 Mar 2026 21:04:21] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000288.root for sample jf17
 [Sun, 15 Mar 2026 21:04:21] INFO Reading /tmp/tmp5wb3xo51/output.35439386._000284.root for sample jf17
 [Sun, 15 Mar 2026 21:04:21] INFO Reading /tmp/tmp5wb3xo51/outpu

...

Processing Batch 69 (Rows 7010336 to 7111324)


 [Sun, 15 Mar 2026 21:09:26] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000018.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000021.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000036.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000061.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000043.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /tmp/tmpjwn8dkmw/output.35439375._000026.root for sample zee
 [Sun, 15 Mar 2026 21:09:27] INFO Reading /media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_EXT0.tap_zee_2sigma_5M_XYZ.root.tgz/user.isilvafe.35439375._000018.XYZ.root.tgz for sample zee
/tmp/ipykernel_986425/4247997998.py:12: DeprecationWarning: Pytho

Processing Batch 70 (Rows 7111324 to 7212137)


 [Sun, 15 Mar 2026 21:09:31] INFO Reading /tmp/tmp173pkqdl/output.35439375._000461.root for sample zee
 [Sun, 15 Mar 2026 21:09:32] INFO Reading /media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_EXT0.tap_zee_2sigma_5M_XYZ.root.tgz/user.isilvafe.35439375._000019.XYZ.root.tgz for sample zee
/tmp/ipykernel_986425/4247997998.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=temp_dir_path)
 [Sun, 15 Mar 2026 21:09:32] INFO Reading /tmp/tmppdzxw0ti/output.35439375._000502.root for sample zee
 [Sun, 15 Mar 2026 21:09:32] INFO Reading /media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/mc21_isabela_qt_2sigma_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_EXT0.

Processing Batch 71 (Rows 7212137 to 7247099)


# Validate Dataset

In [9]:
output_path_glob = output_dir / 'data_*.parquet'

## Check Target

In [19]:
target_query = f"""
SELECT
    COALESCE(CAST(target AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY target
ORDER BY target ASC;
"""

In [20]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(target_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,2177107
1,1,5069992


## Check ID

In [21]:
id_query = f"""
SELECT
    COALESCE(CAST(id AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY id
ORDER BY id ASC;
"""

In [22]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(id_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
...,...,...
7247094,7247094,1
7247095,7247095,1
7247096,7247096,1
7247097,7247097,1


In [23]:
df[df['count'] != 1]

,value,count
